# Spatial Split vs Random Split Comparison

This notebook compares the paper spatial-block protocol (`block10_5seed_mlp1024`)
with the no-spatial random split protocol (`random_5seed_mlp1024`).

It focuses on two questions:

1. How does each model's average task rank change?
2. For each task, how does each model's task score change?

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'results').exists() and (ROOT.parent / 'results').exists():
    ROOT = ROOT.parent.resolve()

SPATIAL_DIR = ROOT / 'results' / 'main_eval'
RANDOM_DIR = ROOT / 'results' / 'random_eval'
SPATIAL_TABLE_DIR = SPATIAL_DIR / 'paper_tables'
RANDOM_TABLE_DIR = RANDOM_DIR / 'paper_tables'
SPATIAL_DIAG_DIR = SPATIAL_DIR / 'diagnostics'
RANDOM_DIAG_DIR = RANDOM_DIR / 'diagnostics'
OUT_DIR = ROOT / 'results' / 'split_comparison'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = ['aether','alphaearth','tessera','satclip','muse','urban2vec','cityfm','calliper','space2vec','place2vec','sphere2vec_fixed']
MODEL_LABEL = {
    'aether':'AETHER',
    'alphaearth':'AlphaEarth',
    'tessera':'TESSERA',
    'calliper':'CaLLiPer',
    'place2vec':'Place2Vec',
    'space2vec':'Space2Vec',
    'sphere2vec_fixed':'PE',
    'satclip':'SatCLIP',
    'cityfm':'CityFM',
    'muse':'MuseCL',
    'urban2vec':'Urban2Vec',
}
TASK_ORDER = ['landuse','road_density','population','age_distribution','gdp','nightlight','pm25','lst_day_mean']
TASK_TITLE = {
    'landuse':'LUC',
    'road_density':'RDE',
    'population':'POP',
    'age_distribution':'AGE*',
    'gdp':'GDP',
    'nightlight':'NTL',
    'pm25':'PM25',
    'lst_day_mean':'LST',
}
TASK_AXIS_LABEL = {
    'landuse':'LUC\nF1 ↑',
    'road_density':'RDE\n$R^2$ ↑',
    'population':'POP\n$R^2$ ↑',
    'age_distribution':'AGE*\nKL ↓',
    'gdp':'GDP\n$R^2$ ↑',
    'nightlight':'NTL\n$R^2$ ↑',
    'pm25':'PM25\n$R^2$ ↑',
    'lst_day_mean':'LST\n$R^2$ ↑',
}
sns.set_theme(style='whitegrid', context='notebook')
MODEL_COLORS = {
    # Matplotlib tab10-style palette requested for dense grouped bars.
    'aether': '#1f77b4',
    'alphaearth': '#ff7f0e',
    'tessera': '#2ca02c',
    'cityfm': '#d62728',
    'calliper': '#9467bd',
    'space2vec': '#8c564b',
    'place2vec': '#e377c2',
    'urban2vec': '#7f7f7f',
    'muse': '#bcbd22',
    'satclip': '#17becf',
    'sphere2vec_fixed': '#5b3a29',
}
PALETTE = {MODEL_LABEL[m]: MODEL_COLORS[m] for m in MODEL_ORDER}

spatial_rank = pd.read_csv(SPATIAL_TABLE_DIR / 'overall_model_average_rank.csv')
random_rank = pd.read_csv(RANDOM_TABLE_DIR / 'overall_model_average_rank.csv')
spatial_primary = pd.read_csv(SPATIAL_TABLE_DIR / 'primary_metric_by_model_task.csv')
random_primary = pd.read_csv(RANDOM_TABLE_DIR / 'primary_metric_by_model_task.csv')
def build_normalized_scores(primary):
    rows = []
    for task in TASK_ORDER:
        sub = primary[primary['task'].astype(str).eq(task)].copy()
        if sub.empty:
            continue
        metric = str(sub['metric'].iloc[0])
        lower = task == 'age_distribution' or metric in {'KL', 'MAE', 'RMSE', 'MSE', 'L1', 'Chebyshev'}
        vals = pd.to_numeric(sub['mean'], errors='coerce')
        lo, hi = vals.min(), vals.max()
        if pd.isna(lo) or pd.isna(hi) or hi == lo:
            sub['score'] = 1.0
        elif lower:
            sub['score'] = (hi - vals) / (hi - lo)
        else:
            sub['score'] = (vals - lo) / (hi - lo)
        rows.append(sub)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

spatial_scores = build_normalized_scores(spatial_primary)
random_scores = build_normalized_scores(random_primary)
spatial_scores.to_csv(SPATIAL_DIAG_DIR / 'normalized_scores_long.csv', index=False)
random_scores.to_csv(RANDOM_DIAG_DIR / 'normalized_scores_long.csv', index=False)

for df in [spatial_rank, random_rank, spatial_primary, random_primary, spatial_scores, random_scores]:
    df['model_label'] = df['model'].map(MODEL_LABEL).fillna(df['model_label'] if 'model_label' in df else df['model'])

print('root:', ROOT)
print('spatial primary rows:', len(spatial_primary), 'random primary rows:', len(random_primary))

## Overall Rank Change

In [ ]:
rank_cmp = (
    spatial_rank[['model','model_label','mean_task_rank','median_task_rank','best_task_rank','worst_task_rank','tasks_scored','coverage_ratio']]
    .rename(columns={
        'mean_task_rank':'spatial_mean_rank',
        'median_task_rank':'spatial_median_rank',
        'best_task_rank':'spatial_best_rank',
        'worst_task_rank':'spatial_worst_rank',
        'tasks_scored':'spatial_tasks_scored',
        'coverage_ratio':'spatial_coverage_ratio',
    })
    .merge(
        random_rank[['model','mean_task_rank','median_task_rank','best_task_rank','worst_task_rank','tasks_scored','coverage_ratio']]
        .rename(columns={
            'mean_task_rank':'random_mean_rank',
            'median_task_rank':'random_median_rank',
            'best_task_rank':'random_best_rank',
            'worst_task_rank':'random_worst_rank',
            'tasks_scored':'random_tasks_scored',
            'coverage_ratio':'random_coverage_ratio',
        }),
        on='model',
        how='outer',
    )
)
rank_cmp['model_label'] = rank_cmp['model'].map(MODEL_LABEL)
rank_cmp['mean_rank_delta_random_minus_spatial'] = rank_cmp['random_mean_rank'] - rank_cmp['spatial_mean_rank']
rank_cmp['abs_mean_rank_delta'] = rank_cmp['mean_rank_delta_random_minus_spatial'].abs()
rank_cmp['model_label'] = pd.Categorical(rank_cmp['model_label'], [MODEL_LABEL[m] for m in MODEL_ORDER], ordered=True)
rank_cmp = rank_cmp.sort_values('model_label')
rank_cmp.to_csv(OUT_DIR / 'split_overall_rank_comparison.csv', index=False)
display(rank_cmp)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.2))
colors = [
    '#2a9d8f' if x < 0 else '#d95f02' if x > 0 else '#777777'
    for x in rank_cmp['mean_rank_delta_random_minus_spatial']
]
axes[0].barh(
    rank_cmp['model_label'].astype(str),
    rank_cmp['mean_rank_delta_random_minus_spatial'],
    color=colors,
)
axes[0].invert_yaxis()
axes[0].axvline(0, color='black', linewidth=0.9)
axes[0].set_title('Average task rank change')
axes[0].set_xlabel('Random split mean rank - spatial split mean rank\nnegative means better rank under random split')
axes[0].set_ylabel('')

rank_long = rank_cmp.melt(
    id_vars=['model','model_label'],
    value_vars=['spatial_mean_rank','random_mean_rank'],
    var_name='protocol',
    value_name='mean_task_rank',
)
rank_long['protocol'] = rank_long['protocol'].map({
    'spatial_mean_rank':'spatial block',
    'random_mean_rank':'random',
})
sns.lineplot(
    data=rank_long,
    x='protocol',
    y='mean_task_rank',
    hue='model_label',
    hue_order=[MODEL_LABEL[m] for m in MODEL_ORDER],
    marker='o',
    palette=PALETTE,
    ax=axes[1],
)
axes[1].invert_yaxis()
axes[1].set_title('Average task rank by protocol')
axes[1].set_xlabel('')
axes[1].set_ylabel('Mean task rank; lower is better')
axes[1].legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
out = FIG_DIR / 'split_overall_rank_change.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
print(out)
display(fig)
plt.close(fig)

## Per-Task Rank Change

In [ ]:
task_rank_cmp = (
    spatial_primary[['task','metric','model','model_label','rank']]
    .rename(columns={'rank':'spatial_rank'})
    .merge(
        random_primary[['task','metric','model','rank']].rename(columns={'rank':'random_rank'}),
        on=['task','metric','model'],
        how='outer',
    )
)
task_rank_cmp['model_label'] = task_rank_cmp['model'].map(MODEL_LABEL)
task_rank_cmp['rank_delta_random_minus_spatial'] = task_rank_cmp['random_rank'] - task_rank_cmp['spatial_rank']
task_rank_cmp['task'] = pd.Categorical(task_rank_cmp['task'], TASK_ORDER, ordered=True)
task_rank_cmp['model_label'] = pd.Categorical(task_rank_cmp['model_label'], [MODEL_LABEL[m] for m in MODEL_ORDER], ordered=True)
task_rank_cmp = task_rank_cmp.sort_values(['task','model_label']).reset_index(drop=True)
task_rank_cmp.to_csv(OUT_DIR / 'split_task_rank_delta.csv', index=False)
display(task_rank_cmp)

rank_delta_heat = task_rank_cmp.pivot(index='model_label', columns='task', values='rank_delta_random_minus_spatial').reindex(
    index=[MODEL_LABEL[m] for m in MODEL_ORDER],
    columns=TASK_ORDER,
)
fig, ax = plt.subplots(figsize=(11.5, 5.4))
vmax = np.nanmax(np.abs(rank_delta_heat.to_numpy(dtype=float)))
sns.heatmap(
    rank_delta_heat,
    annot=True,
    fmt='.0f',
    cmap='RdBu_r',
    center=0,
    vmin=-vmax,
    vmax=vmax,
    ax=ax,
)
ax.set_title('Task rank change: random split - spatial split')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticklabels([TASK_AXIS_LABEL[t] for t in TASK_ORDER], rotation=35, ha='right')
fig.tight_layout()
out = FIG_DIR / 'split_task_rank_delta_heatmap.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
print(out)
display(fig)
plt.close(fig)

## Raw Primary Metric Change

In [ ]:
raw_cmp = (
    spatial_primary[['task','metric','higher_is_better','model','model_label','mean','rank']]
    .rename(columns={'mean':'spatial_mean','rank':'spatial_rank'})
    .merge(
        random_primary[['task','metric','model','mean','rank']].rename(columns={'mean':'random_mean','rank':'random_rank'}),
        on=['task','metric','model'],
        how='outer',
    )
)
raw_cmp['model_label'] = raw_cmp['model'].map(MODEL_LABEL)
raw_cmp['task_title'] = raw_cmp['task'].map(TASK_TITLE)
raw_cmp['raw_delta_random_minus_spatial'] = raw_cmp['random_mean'] - raw_cmp['spatial_mean']
raw_cmp['task'] = pd.Categorical(raw_cmp['task'], TASK_ORDER, ordered=True)
raw_cmp['model_label'] = pd.Categorical(raw_cmp['model_label'], [MODEL_LABEL[m] for m in MODEL_ORDER], ordered=True)
raw_cmp = raw_cmp.sort_values(['task','model_label']).reset_index(drop=True)
raw_cmp.to_csv(OUT_DIR / 'split_task_primary_metric_delta.csv', index=False)
display(raw_cmp)

raw_delta_heat = raw_cmp.pivot(index='model_label', columns='task', values='raw_delta_random_minus_spatial').reindex(
    index=[MODEL_LABEL[m] for m in MODEL_ORDER],
    columns=TASK_ORDER,
)
fig, ax = plt.subplots(figsize=(11.5, 5.4))
vmax = np.nanmax(np.abs(raw_delta_heat.to_numpy(dtype=float)))
sns.heatmap(
    raw_delta_heat,
    annot=True,
    fmt='.3f',
    cmap='RdBu_r',
    center=0,
    vmin=-vmax,
    vmax=vmax,
    ax=ax,
)
ax.set_title('Raw primary metric change: random split - spatial split')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticklabels([TASK_AXIS_LABEL[t] for t in TASK_ORDER], rotation=35, ha='right')
fig.tight_layout()
out = FIG_DIR / 'split_task_raw_primary_metric_delta_heatmap.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
print(out)
display(fig)
plt.close(fig)

In [ ]:
score_cmp = (
    spatial_scores[['task','metric','model','model_label','score']]
    .rename(columns={'score':'spatial_score'})
    .merge(
        random_scores[['task','metric','model','score']].rename(columns={'score':'random_score'}),
        on=['task','metric','model'],
        how='outer',
    )
)
score_cmp['model_label'] = score_cmp['model'].map(MODEL_LABEL)
score_cmp['score_delta_random_minus_spatial'] = score_cmp['random_score'] - score_cmp['spatial_score']
score_cmp['task'] = pd.Categorical(score_cmp['task'], TASK_ORDER, ordered=True)
score_cmp['model_label'] = pd.Categorical(score_cmp['model_label'], [MODEL_LABEL[m] for m in MODEL_ORDER], ordered=True)
score_cmp = score_cmp.sort_values(['task','model_label']).reset_index(drop=True)
score_cmp.to_csv(OUT_DIR / 'split_task_normalized_score_delta.csv', index=False)
display(score_cmp)

score_delta_heat = score_cmp.pivot(index='model_label', columns='task', values='score_delta_random_minus_spatial').reindex(
    index=[MODEL_LABEL[m] for m in MODEL_ORDER],
    columns=TASK_ORDER,
)
fig, ax = plt.subplots(figsize=(11.5, 5.4))
vmax = np.nanmax(np.abs(score_delta_heat.to_numpy(dtype=float)))
sns.heatmap(
    score_delta_heat,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-vmax,
    vmax=vmax,
    ax=ax,
)
ax.set_title('Normalized primary score change: random split - spatial split')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticklabels([TASK_AXIS_LABEL[t] for t in TASK_ORDER], rotation=35, ha='right')
fig.tight_layout()
out = FIG_DIR / 'split_task_normalized_score_delta_heatmap.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
print(out)
display(fig)
plt.close(fig)

fig, axes = plt.subplots(2, 4, figsize=(20, 8.5), sharey=True)
axes = axes.ravel()
for ax, task in zip(axes, TASK_ORDER):
    sub = score_cmp[score_cmp['task'].astype(str).eq(task)].copy()
    sub['model_label'] = pd.Categorical(sub['model_label'], [MODEL_LABEL[m] for m in MODEL_ORDER], ordered=True)
    sns.barplot(
        data=sub,
        x='model_label',
        y='score_delta_random_minus_spatial',
        hue='model_label',
        order=[MODEL_LABEL[m] for m in MODEL_ORDER],
        hue_order=[MODEL_LABEL[m] for m in MODEL_ORDER],
        palette=PALETTE,
        legend=False,
        ax=ax,
    )
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(TASK_AXIS_LABEL[task].replace('\n', ' '))
    ax.set_xlabel('')
    ax.set_ylabel('Random - spatial normalized score' if ax is axes[0] else '')
    ax.tick_params(axis='x', rotation=70, labelsize=8)
fig.tight_layout()
out = FIG_DIR / 'split_task_normalized_score_delta_bars.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
print(out)
display(fig)
plt.close(fig)

largest_score = score_cmp.assign(abs_delta=score_cmp['score_delta_random_minus_spatial'].abs()).sort_values('abs_delta', ascending=False).head(30)
largest_score.to_csv(OUT_DIR / 'split_largest_normalized_score_changes.csv', index=False)
display(largest_score)

largest_rank = task_rank_cmp.assign(abs_delta=task_rank_cmp['rank_delta_random_minus_spatial'].abs()).sort_values('abs_delta', ascending=False).head(30)
largest_rank.to_csv(OUT_DIR / 'split_largest_task_rank_changes.csv', index=False)
display(largest_rank)
